# Capability — Single-Image In-Context Learning
Guide the model with one exemplar image before detecting the same object in a different scene.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericpence/perceptron_repo/blob/main/cookbook/recipes/capabilities/in-context-learning/in-context-learning.ipynb)

## Install dependencies
Install the SDK plus helpers for drawing overlays inside the notebook session.

In [ ]:
!uv pip install --upgrade perceptron pillow

## Configure the Perceptron client
Load the API key (inline or from the environment) once, then reuse the configured client for the rest of the notebook.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

from IPython.display import Image as IPyImage, display
from PIL import Image, ImageDraw, ImageFont

from perceptron import annotate_image, bbox, configure, detect
from perceptron.pointing.geometry import scale_box_to_pixels

PERCEPTRON_API_KEY = os.environ.get("PERCEPTRON_API_KEY", "<your Perceptron API key>")

configure(
    provider="perceptron",
    api_key=PERCEPTRON_API_KEY,
)

BASE_URL = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/in-context-learning/single/"
EXAMPLE_IMAGE = Path("cake_mixer_example.webp")
TARGET_IMAGE = Path("find_kitchen_item.webp")
ANNOTATED_PATH = Path("find_kitchen_item_annotated.png")

for filename, path_obj in (("cake_mixer_example.webp", EXAMPLE_IMAGE), ("find_kitchen_item.webp", TARGET_IMAGE)):
    if not path_obj.exists():
        urlretrieve(BASE_URL + filename, path_obj)


> Exemplar and target shots download automatically from `cookbook/_shared/assets/in-context-learning/single/`. `scale_box_to_pixels` converts Perceptron's 1–1000 geometry back to pixels before drawing overlays.

## Bootstrap the exemplar box
Run one detection pass on the exemplar image so we can feed a precise bounding box back as context.

In [ ]:
bootstrap = detect(
    str(EXAMPLE_IMAGE),
    classes=["objectCategory1"],
    max_outputs=1,
)
if not bootstrap.points:
    raise RuntimeError("Detect returned no boxes for the exemplar. Adjust the label or image.")

first_box = bootstrap.points[0]
example_shot = annotate_image(
    str(EXAMPLE_IMAGE),
    {
        "objectCategory1": [
            bbox(
                int(first_box.top_left.x),
                int(first_box.top_left.y),
                int(first_box.bottom_right.x),
                int(first_box.bottom_right.y),
                mention="objectCategory1",
            )
        ]
    },
)
collections = example_shot.get("collections") or []
boxes = example_shot.get("boxes") or []
placeholder = "objectCategory1"
if collections:
    exemplar_annotation = collections[0]
elif boxes:
    exemplar_annotation = boxes[0]
else:
    raise RuntimeError("annotate_image returned no collections or boxes; check the exemplar annotations.")
print("Prepared exemplar guidance with", getattr(exemplar_annotation, "mention", placeholder) or placeholder)

## Detect the same object in a new scene
Pass the exemplar annotation back to `detect` via the `examples` argument to nudge the model toward consistent grounding.

In [ ]:
result = detect(
    str(TARGET_IMAGE),
    classes=["objectCategory1"],
    examples=[example_shot],
)

print(result.text)
boxes = result.points or []
print(f"Returned {len(boxes)} grounded regions")

## Render the grounded output
Overlay the predicted boxes on the target scene and preview the saved PNG inline.

In [ ]:
img = Image.open(TARGET_IMAGE).convert("RGB")
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype("arial.ttf", size=20)
except OSError:
    font = ImageFont.load_default()

if boxes:
    for box in boxes:
        scaled = scale_box_to_pixels(box, width=img.width, height=img.height)
        top_left = scaled.top_left
        bottom_right = scaled.bottom_right
        tlx, tly = int(round(top_left.x)), int(round(top_left.y))
        brx, bry = int(round(bottom_right.x)), int(round(bottom_right.y))
        draw.rectangle([tlx, tly, brx, bry], outline="lime", width=3)
        label = box.mention or getattr(box, "label", None) or "objectCategory1"
        text_position = (tlx, max(tly - 20, 0))
        draw.text(text_position, label, fill="lime", font=font)
else:
    print("No boxes returned; adjust the exemplar or label and rerun.")

img.save(ANNOTATED_PATH)
display(IPyImage(filename=str(ANNOTATED_PATH)))
print(f"Saved annotated target to {ANNOTATED_PATH}")


## Conclusion & next steps
- Swap in different exemplar / target pairs to explore other categories.
- Add more than one exemplar by appending to the `examples` list for tougher distinctions.
- Combine this flow with detection or Q&A helpers to build end-to-end pipelines.